# 04 - Diagnostics and Figures  (Stage 4)

**Purpose.** Produce the pathway map, seasonal & interannual fractions, transit
time distributions, centroid map (day 50 / 100 / 150), and the pathway summary
table for the chosen `k`.

**Input.** `data/labeled_trajectories.parquet` (Stage 3), the chosen k-means
model, the saved `scaler.pkl`, and the original Zarr stores (for full trajectory
paths / transit times).
**Output.** PNGs in `figures/` and a printed summary table.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # project root: config.py, pipeline.py
import numpy as np
import pandas as pd
import config as C
import pipeline as P
print("project root:", C.PROJECT_ROOT)
print("sampling days:", C.DAYS, "| feature space:", C.FEATURE_SPACE)
import cartopy.crs as ccrs, cartopy.feature as cfeature
from matplotlib.lines import Line2D
import xarray as xr

project root: /work/bk1450/b383184/Amazon/Mercator/notebooks/Analysis/kmeans_analysis/kmean_analysis_50_150_stdZ
sampling days: [30, 50, 100, 150] | feature space: zscore


**Raw clusters vs. merged groups.** Set `LABEL_COL = "cluster_group"` to make
every figure at the merged-pathway level (requires a `GROUP_MAP` in `config`),
or `"cluster_label"` for the raw k-means clusters. Everything below keys off
`LABEL_COL`, so the figures, fractions and table all follow your choice.

In [2]:
BEST_K = 40                    # <-- must match notebook 03 (papermill: -p BEST_K <k>)
LABEL_COL = "cluster_group"    # "cluster_group" (merged) or "cluster_label" (raw)

In [3]:
# Parameters
BEST_K = 50
LABEL_COL = "cluster_group"


In [4]:
import pickle, matplotlib.pyplot as plt
lab = pd.read_parquet(C.LABELED_FILE)
with open(C.MODELS_DIR / f"kmeans_k{BEST_K}.pkl", "rb") as f:
    km = pickle.load(f)
with open(C.MODELS_DIR / "scaler.pkl", "rb") as f:
    scaler = pickle.load(f)
# Per-day weighting the model was fit in (config.DAY_WEIGHTS). Centroids live in
# WEIGHTED standardized space, so divide by W to undo the weighting BEFORE
# inverse_transform gives back real degrees (mirrors notebook 02).
W = P.feature_weight_vector()
labelled = lab[lab[LABEL_COL] >= 0].copy()
n_lab = int(labelled[LABEL_COL].max()) + 1
cmap = plt.get_cmap("tab20", max(n_lab, 3))

# Centroids in real degrees: [lat50, lon50, lat100, lon100, lat150, lon150] per label.
# (inverse_transform of the standardized k-means centres, via the saved scaler).
raw_cent = P.centroids_to_degrees(km.cluster_centers_ / W, scaler)
if LABEL_COL == "cluster_group" and C.GROUP_MAP:
    _, raw2grp = P.apply_group_map(np.arange(BEST_K), C.GROUP_MAP, BEST_K)
    sizes = lab.loc[lab.cluster_label >= 0, "cluster_label"].value_counts()
    # Merge in the WEIGHTED standardized space (size-weighted mean of the
    # weighted centres), then divide by W and inverse-transform -> a proper
    # group centroid in degrees.
    dim = km.cluster_centers_.shape[1]
    grp_centers = np.zeros((n_lab, dim)); wsum = np.zeros(n_lab)
    for c in range(BEST_K):
        g = raw2grp[c]; w = float(sizes.get(c, 0))
        grp_centers[g] += km.cluster_centers_[c] * w; wsum[g] += w
    grp_centers /= np.where(wsum[:, None] == 0, 1, wsum[:, None])
    disp_cent = P.centroids_to_degrees(grp_centers / W, scaler)
else:
    disp_cent = raw_cent
LABEL_NAMES = C.GROUP_NAMES if LABEL_COL == "cluster_group" else {}
print(f"labelled trajectories: {len(labelled):,} | {LABEL_COL}: {n_lab} labels")

labelled trajectories: 16,160,000 | cluster_group: 16 labels


## Figure 1 - Pathway map

Plot up to ~5,000 *full* trajectories per cluster, coloured by label. Full paths
are streamed from the Zarr stores so we never hold everything in memory. We pick
a random set of `trajectory_id`s per cluster, group them by store, and read only
those rows.

In [5]:
rng = np.random.default_rng(C.RANDOM_STATE)

# choose <=5000 trajectory_ids per label (raw cluster or merged group)
pick = (labelled.groupby(LABEL_COL, group_keys=False)
        .apply(lambda d: d.sample(min(5000, len(d)), random_state=C.RANDOM_STATE)))
pick = pick[["trajectory_id", LABEL_COL]].rename(columns={LABEL_COL: "lab"}).copy()
pick["store_index"] = pick.trajectory_id // C.TRAJ_PER_STORE
pick["local"] = pick.trajectory_id % C.TRAJ_PER_STORE
stores = C.list_stores()

/tmp/ipykernel_765546/1928533925.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda d: d.sample(min(5000, len(d)), random_state=C.RANDOM_STATE)))


In [6]:
fig = plt.figure(figsize=(11, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([C.DOMAIN["lon_min"], C.DOMAIN["lon_max"],
               C.DOMAIN["lat_min"], C.DOMAIN["lat_max"]], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="0.85"); ax.coastlines(lw=.5)

for si, grp in pick.groupby("store_index"):
    ds = xr.open_zarr(stores[si])
    loc = grp.local.to_numpy()
    lons = ds.lon.values[loc]; lats = ds.lat.values[loc]
    cl = grp.lab.to_numpy()
    for j in range(len(loc)):
        ax.plot(lons[j], lats[j], color=cmap(cl[j]), lw=.2, alpha=.3,
                transform=ccrs.PlateCarree())
ax.set_title(f"Plume pathways by {LABEL_COL} (k={BEST_K}, {n_lab} labels)")
# colour legend: one solid swatch per label (full opacity so it's readable even
# though the trajectories are drawn thin/transparent).
handles = [Line2D([0], [0], color=cmap(c), lw=3,
                  label=f"{LABEL_NAMES.get(c, c)} (n={int((pick.lab == c).sum())})")
           for c in range(n_lab)]
ax.legend(handles=handles, title=LABEL_COL, fontsize=7, ncol=1,
          loc="upper left", bbox_to_anchor=(1.01, 1))
fig.savefig(C.FIG_DIR / f"pathway_map_k{BEST_K}_{LABEL_COL}.png", dpi=140, bbox_inches="tight")
plt.show()

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shape

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shape

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shape

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shape

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shape

...

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shape

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shape

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shape

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shape

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)


## Figure 1b - Pathway map, one panel per label

The combined map above overlaps every pathway; here the **same** sampled
trajectories are split so each `LABEL_COL` (cluster or merged group) gets its own
panel. All panels share the map `DOMAIN` extent so they can be compared directly,
and each panel keeps its label's colour (title tinted to match).

In [7]:
# Small multiples: each panel shows only the trajectories assigned to that label,
# so overlapping pathways stay readable (same `pick` sample and colours as Fig 1).
# Shared extent = the map DOMAIN so panels are directly comparable. We make ONE
# pass over the Zarr stores and route each trajectory to its label's panel.
ncol = min(n_lab, 4)
nrow = int(np.ceil(n_lab / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.4 * nrow),
                         subplot_kw={"projection": ccrs.PlateCarree()}, squeeze=False)
axflat = axes.ravel()
for c in range(n_lab):
    ax = axflat[c]
    ax.set_extent([C.DOMAIN["lon_min"], C.DOMAIN["lon_max"],
                   C.DOMAIN["lat_min"], C.DOMAIN["lat_max"]], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="0.85"); ax.coastlines(lw=.5)
    ax.set_title(f"{LABEL_NAMES.get(c, c)} (n={int((pick.lab == c).sum())})",
                 fontsize=9, color=cmap(c), fontweight="bold")

for si, grp in pick.groupby("store_index"):
    ds = xr.open_zarr(stores[si])
    loc = grp.local.to_numpy()
    lons = ds.lon.values[loc]; lats = ds.lat.values[loc]
    cl = grp.lab.to_numpy()
    for j in range(len(loc)):
        axflat[cl[j]].plot(lons[j], lats[j], color=cmap(cl[j]), lw=.2, alpha=.3,
                           transform=ccrs.PlateCarree())

for k in range(n_lab, nrow * ncol):       # blank any unused panels
    axflat[k].axis("off")
fig.suptitle(f"Pathways per {LABEL_COL} (k={BEST_K}, {n_lab} labels)", y=1.0, fontsize=12)
fig.tight_layout()
fig.savefig(C.FIG_DIR / f"pathway_maps_by_{LABEL_COL}_k{BEST_K}.png", dpi=130, bbox_inches="tight")
plt.show()

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shape

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shape

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shape

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shape

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shape

...

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shape

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shape

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shape

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)
/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shape

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/shapely/creation.py:218: RuntimeWarning: invalid value encountered in linestrings
  return lib.linestrings(coords, np.intc(handle_nan), out=out, **kwargs)


## Figure 2 - Seasonal pathway fractions (stacked bars by release month)

In [8]:
def label_name(c):
    return LABEL_NAMES.get(c, str(c))

def fraction_table(group_col):
    g = labelled.groupby([group_col, LABEL_COL]).size().unstack(fill_value=0)
    return g.div(g.sum(axis=1), axis=0) * 100

seas = fraction_table("release_month").reindex(range(1, 13))
fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(seas))
for c in range(n_lab):
    vals = seas.get(c, pd.Series(0, index=seas.index)).to_numpy()
    ax.bar(seas.index, vals, bottom=bottom, color=cmap(c), label=label_name(c))
    bottom += vals
ax.set_xlabel("release month"); ax.set_ylabel("% of particles")
ax.set_title(f"Seasonal pathway fractions ({LABEL_COL})")
ax.legend(ncol=2, fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")
fig.savefig(C.FIG_DIR / f"seasonal_fractions_k{BEST_K}_{LABEL_COL}.png", dpi=130, bbox_inches="tight")
plt.show()

## Figure 3 - Interannual pathway fractions (stacked area by release year)

In [9]:
yr = fraction_table("release_year").sort_index()
fig, ax = plt.subplots(figsize=(11, 5))
ax.stackplot(yr.index, *[yr.get(c, pd.Series(0, index=yr.index)).to_numpy()
                         for c in range(n_lab)],
             colors=[cmap(c) for c in range(n_lab)], labels=[label_name(c) for c in range(n_lab)])
ax.set_xlabel("release year"); ax.set_ylabel("% of particles"); ax.set_ylim(0, 100)
ax.set_title(f"Interannual pathway fractions ({LABEL_COL})")
ax.legend(ncol=2, fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")
fig.savefig(C.FIG_DIR / f"interannual_fractions_k{BEST_K}_{LABEL_COL}.png", dpi=130, bbox_inches="tight")
plt.show()

## Figure 4 - Transit-time distributions per cluster

Time (days since release) to first reach 10°N, computed from the Zarr stores via
`pipeline.first_crossing_days`. We reuse the per-cluster `pick` sample from
Figure 1 to keep the I/O bounded.

In [10]:
transit = {}   # trajectory_id -> days to 10N
for si, grp in pick.groupby("store_index"):
    days = P.first_crossing_days(stores[si], lat_thresh=10.0)
    for tid, loc in zip(grp.trajectory_id, grp.local):
        transit[tid] = days[loc]
pick["t10N"] = pick.trajectory_id.map(transit)

ncol = 5; nrow = int(np.ceil(n_lab / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3*ncol, 2.4*nrow), squeeze=False)
for c in range(n_lab):
    ax = axes[c // ncol][c % ncol]
    d = pick.loc[pick.lab == c, "t10N"].dropna()
    ax.hist(d, bins=30, color=cmap(c))
    ax.set_title(f"{label_name(c)} (n={len(d)})", fontsize=8)
    ax.set_xlabel("days to 10N", fontsize=7)
for j in range(n_lab, nrow*ncol):
    axes[j // ncol][j % ncol].axis("off")
fig.tight_layout()
fig.savefig(C.FIG_DIR / f"transit_time_histograms_k{BEST_K}_{LABEL_COL}.png", dpi=120)
plt.show()

## Figure 5 - Centroid positions (marker per sampling day, joined by a line)

In [11]:
cent = disp_cent   # per label: [lat_d0, lon_d0, lat_d1, lon_d1, ...] real degrees
pct = labelled[LABEL_COL].value_counts(normalize=True).sort_index() * 100
day_markers = ["o", "s", "*", "^", "D", "P"]
day_sizes   = [90, 120, 240, 150, 150, 150]
fig = plt.figure(figsize=(11, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([C.DOMAIN["lon_min"], C.DOMAIN["lon_max"],
               C.DOMAIN["lat_min"], C.DOMAIN["lat_max"]], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="0.85"); ax.coastlines(lw=.5)
for c in range(n_lab):
    lats = cent[c, 0::2]; lons = cent[c, 1::2]      # one entry per day in C.DAYS
    ax.plot(lons, lats, "-", color=cmap(c), lw=1, alpha=.6, transform=ccrs.PlateCarree())
    for j in range(len(C.DAYS)):
        ax.scatter(lons[j], lats[j], color=cmap(c), s=day_sizes[j % len(day_sizes)],
                   marker=day_markers[j % len(day_markers)], edgecolor="k",
                   transform=ccrs.PlateCarree())
    ax.text(lons[-1], lats[-1], f" {label_name(c)} ({pct.get(c,0):.0f}%)", fontsize=8,
            transform=ccrs.PlateCarree())
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], marker=day_markers[j % len(day_markers)], color="k",
                  ls="", mfc="0.7", label=f"day {d}") for j, d in enumerate(C.DAYS)]
ax.legend(handles=handles, loc="lower right", fontsize=8)
ax.set_title(f"Centroids ({LABEL_COL}, k={BEST_K})  days: {list(C.DAYS)}")
fig.savefig(C.FIG_DIR / f"centroids_k{BEST_K}_{LABEL_COL}.png", dpi=140, bbox_inches="tight")
plt.show()

## Table 1 - Pathway summary

In [12]:
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
last_day = C.DAYS[-1]
rows = []
for c in range(n_lab):
    sel = labelled[labelled[LABEL_COL] == c]
    peak = month_names[int(sel.release_month.mode().iloc[0]) - 1] if len(sel) else "-"
    mt = pick.loc[pick.lab == c, "t10N"].dropna()
    rows.append(dict(
        label=c,
        name=LABEL_NAMES.get(c, ""),
        pct=round(len(sel) / len(labelled) * 100, 1),
        peak_month=peak,
        mean_transit_10N=round(float(mt.mean()), 1) if len(mt) else np.nan,
        mean_lat_last=round(float(sel[f"lat_{last_day}"].mean()), 2),
        mean_lon_last=round(float(sel[f"lon_{last_day}"].mean()), 2),
    ))
summary = pd.DataFrame(rows)
summary.to_csv(C.DATA_DIR / f"pathway_summary_k{BEST_K}_{LABEL_COL}.csv", index=False)
print(f"(mean_lat_last / mean_lon_last are the day-{last_day} positions)")
print(summary.to_string(index=False))

(mean_lat_last / mean_lon_last are the day-150 positions)
 label name  pct peak_month  mean_transit_10N  mean_lat_last  mean_lon_last
     0    A  8.3        Mar              83.6          11.61         -62.32
     1    B 44.3        May              58.3           5.25         -53.82
     2    C  3.0        Nov              48.7          14.77         -69.32
     3    D  5.4        Aug              51.1          13.00         -60.71
     4    E  7.5        Sep              48.6          11.29         -60.48
     5    F 20.2        Jan             108.2           4.18         -52.41
     6       0.8        Nov              68.4          10.69         -61.37
     7       2.0        Aug              45.5          14.57         -63.94
     8       1.0        Oct              59.0          15.09         -78.54
     9       0.3        Jul              59.8           8.59         -43.86
    10       1.5        Jun              81.6           6.05         -53.53
    11       1.7        Jan   